# GPU Profiling: Base vs Control vs Runtime-Aware

Profiles all three models with the same setup (same hardware, batch size, and input) using PyTorch Profiler.
Fine-tuned adapters are merged into the base model before profiling.

In [7]:
import os
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get('EFFICIENT_CODEGEN_ROOT', '/workspace/efficient-codegen'))
os.chdir(PROJECT_ROOT)

BASE_MODEL      = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
CONTROL_ADAPTER = str(PROJECT_ROOT / 'checkpoints/control')
RUNTIME_ADAPTER = str(PROJECT_ROOT / 'checkpoints/runtime_aware')

CONTROL_MERGED  = str(PROJECT_ROOT / 'checkpoints/control_merged')
RUNTIME_MERGED  = str(PROJECT_ROOT / 'checkpoints/runtime_aware_merged')

PROFILE_INPUT   = 'data/curated/scale1k/dataset_clean.json'
BATCH_SIZE      = 64
MAX_NEW_TOKENS  = 128
LIMIT           = 320

WANDB_PROJECT   = 'hpml-efficient-codegen'
WANDB_ENTITY    = 'efficient-codegen'
USE_WANDB       = True

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BASE_MODEL:  ', BASE_MODEL)
print('CONTROL_ADAPTER:', CONTROL_ADAPTER)
print('RUNTIME_ADAPTER:', RUNTIME_ADAPTER)
print('USE_WANDB:', USE_WANDB)


PROJECT_ROOT: /workspace/efficient-codegen
BASE_MODEL:   Qwen/Qwen2.5-Coder-1.5B-Instruct
CONTROL_ADAPTER: /workspace/efficient-codegen/checkpoints/control
RUNTIME_ADAPTER: /workspace/efficient-codegen/checkpoints/runtime_aware
USE_WANDB: True


In [ ]:
import wandb
import os

os.environ["WANDB_API_KEY"] = ""  # paste your key here, or set it as an env var
if os.environ.get("WANDB_API_KEY"):
    wandb.login(key=os.environ["WANDB_API_KEY"])
else:
    os.environ.setdefault("WANDB_MODE", "disabled")
    USE_WANDB = False
    print("WANDB_API_KEY not set — W&B logging disabled.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


## Step 1: Merge LoRA Adapters

Fine-tuned checkpoints are LoRA adapters — merge them into the base model weights before profiling.
Skip if merged checkpoints already exist.

In [9]:
def merge_adapter(adapter_path, output_dir):
    if Path(output_dir).exists():
        print(f'Already merged: {output_dir}')
        return
    print(f'Merging {adapter_path} -> {output_dir}')
    subprocess.run([
        sys.executable, 'serving/merge_checkpoint.py',
        '--adapter_path',    adapter_path,
        '--base_model_name', BASE_MODEL,
        '--output_dir',      output_dir,
    ], check=True)
    print('Done.')

merge_adapter(CONTROL_ADAPTER, CONTROL_MERGED)
merge_adapter(RUNTIME_ADAPTER, RUNTIME_MERGED)

Already merged: /workspace/efficient-codegen/checkpoints/control_merged
Already merged: /workspace/efficient-codegen/checkpoints/runtime_aware_merged


## Step 2: Profile Each Model

Runs `profile_model.py` for each model with identical settings.
Traces are saved to `outputs/gpu_profiling/<model>/`.

In [10]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [11]:
MODELS = {
    'base':          BASE_MODEL,
    'control':       CONTROL_MERGED,
    'runtime_aware': RUNTIME_MERGED,
}

TRACE_BASE = PROJECT_ROOT / 'outputs/gpu_profiling'

summaries = {}

for label, model_path in MODELS.items():
    trace_dir = str(TRACE_BASE / label)
    print(f'\n{"="*60}')
    print(f'Profiling: {label}')
    print(f'{"="*60}')
    cmd = [
        sys.executable, 'profiling/profile_model.py',
        '--input_path',     PROFILE_INPUT,
        '--model_name',     model_path,
        '--trace_dir',      trace_dir,
        '--limit',          str(LIMIT),
        '--batch_size',     str(BATCH_SIZE),
        '--max_new_tokens', str(MAX_NEW_TOKENS),
        '--wandb_project',  WANDB_PROJECT,
        '--wandb_entity',   WANDB_ENTITY,
        '--wandb_run_name', f'gpu-profiling-{label}',
    ]
    if USE_WANDB:
        cmd.append('--use_wandb')
    subprocess.run(cmd, check=True)
    summaries[label] = {'trace_dir': trace_dir}



Profiling: base


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: yz3202 (efficient-codegen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /workspace/efficient-codegen/wandb/run-20260425_222155-akowzxnq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run gpu-profiling-base
wandb: ⭐️ View project at https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: 🚀 View run at https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/akowzxnq
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 403.09it/s]
/workspace/efficient-codegen/profiling/profile_model.py:181: UserWarning: Profiler won't be using warmup, this can skew profiler results
  schedule=schedule(wait=0, warmup=0, active=1),
/venv/main/lib/python3.12/site-packages/torch/profiler/pr

[batch 1/1] batch_size=20 input_len=738 output_len=128 latency=7.5944s tok/s=337.1

Summary:
Average latency:      7.594430367462337
Average input length: 738.0
Average output len:   128.0
Trace directory:      /workspace/efficient-codegen/outputs/gpu_profiling/base
Peak CUDA memory MB:  4307.15


wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 0-0, summary, console lines 7-14
wandb: 
wandb: Run history:
wandb:       avg_input_len ▁
wandb:       avg_latency_s ▁
wandb:      avg_output_len ▁
wandb: peak_cuda_memory_mb ▁
wandb: 
wandb: Run summary:
wandb:       avg_input_len 738
wandb:       avg_latency_s 7.59443
wandb:      avg_output_len 128
wandb: peak_cuda_memory_mb 4307.15039
wandb: 
wandb: 🚀 View run gpu-profiling-base at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/akowzxnq
wandb: ⭐️ View project at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260425_222155-akowzxnq/logs



Profiling: control


...

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:       avg_input_len ▁
wandb:       avg_latency_s ▁
wandb:      avg_output_len ▁
wandb: peak_cuda_memory_mb ▁
wandb: 
wandb: Run summary:
wandb:       avg_input_len 738
wandb:       avg_latency_s 7.6811
wandb:      avg_output_len 128
wandb: peak_cuda_memory_mb 4307.15039
wandb: 
wandb: 🚀 View run gpu-profiling-control at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/7st3ofwj
wandb: ⭐️ View project at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260425_222240-7st3ofwj/logs



Profiling: runtime_aware


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: yz3202 (efficient-codegen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /workspace/efficient-codegen/wandb/run-20260425_222323-qiuf8au7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run gpu-profiling-runtime_aware
wandb: ⭐️ View project at https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: 🚀 View run at https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/qiuf8au7
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 618.42it/s]
/workspace/efficient-codegen/profiling/profile_model.py:181: UserWarning: Profiler won't be using warmup, this can skew profiler results
  schedule=schedule(wait=0, warmup=0, active=1),
/venv/main/lib/python3.12/site-packages/torch/pr

[batch 1/1] batch_size=20 input_len=738 output_len=128 latency=7.4462s tok/s=343.8

Summary:
Average latency:      7.446197031997144
Average input length: 738.0
Average output len:   128.0
Trace directory:      /workspace/efficient-codegen/outputs/gpu_profiling/runtime_aware
Peak CUDA memory MB:  4307.15


wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:       avg_input_len ▁
wandb:       avg_latency_s ▁
wandb:      avg_output_len ▁
wandb: peak_cuda_memory_mb ▁
wandb: 
wandb: Run summary:
wandb:       avg_input_len 738
wandb:       avg_latency_s 7.4462
wandb:      avg_output_len 128
wandb: peak_cuda_memory_mb 4307.15039
wandb: 
wandb: 🚀 View run gpu-profiling-runtime_aware at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen/runs/qiuf8au7
wandb: ⭐️ View project at: https://wandb.ai/efficient-codegen/hpml-efficient-codegen
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260425_222323-qiuf8au7/logs


## Step 3: Parse Traces and Compare

In [12]:
import json
from collections import defaultdict
import pandas as pd

TOP_N = 15
REPORT = {}

for label in MODELS:
    trace_dir = TRACE_BASE / label
    trace_files = sorted(trace_dir.rglob('*.pt.trace.json')) if trace_dir.exists() else []
    if not trace_files:
        print(f'No traces found for {label}')
        continue

    trace_file = max(trace_files, key=lambda f: f.stat().st_size)
    print(f'\n[{label}] Parsing: {trace_file.name}')

    with open(trace_file) as f:
        data = json.load(f)

    events = data.get('traceEvents', data) if isinstance(data, dict) else data
    op_time = defaultdict(float)
    op_calls = defaultdict(int)
    total = 0.0
    for e in events:
        if not isinstance(e, dict):
            continue
        if e.get('ph') == 'X' and e.get('dur', 0) > 0:
            name = e.get('name', 'unknown')
            dur_ms = e['dur'] / 1000
            op_time[name] += dur_ms
            op_calls[name] += 1
            total += dur_ms

    top = sorted(op_time.items(), key=lambda x: x[1], reverse=True)[:TOP_N]
    REPORT[label] = {'total_ms': total, 'top_ops': top, 'op_time': op_time, 'op_calls': op_calls}

    print(f"  Total traced time: {total:.2f} ms")
    print(f"  {'Operator':<50} {'ms':>10} {'%':>7}")
    print(f"  {'-'*70}")
    for name, ms in top:
        pct = ms / total * 100
        print(f"  {name:<50} {ms:>10.2f} {pct:>6.1f}%")


[base] Parsing: worker0.1777155743127388248.pt.trace.json
  Total traced time: 39239.66 ms
  Operator                                                   ms       %
  ----------------------------------------------------------------------
  ProfilerStep#0                                       15215.09   38.8%
  PyTorch Profiler (0)                                  7620.85   19.4%
  aten::linear                                          1799.06    4.6%
  cudaLaunchKernel                                      1478.61    3.8%
  aten::matmul                                           966.54    2.5%
  aten::mm                                               737.12    1.9%
  void at::native::elementwise_kernel<128, 4, at::native::gpu_kernel_impl_nocast<at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda()#3}::operator()() const::{lambda()#12}::operator()() const::{lambda(c10::BFloat16)#1}>(at::TensorIteratorBase&, at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lamb

## Step 4: Side-by-Side Comparison

Top operators compared across all three models.

In [13]:
KEY_OPS = [
    'aten::linear',
    'aten::scaled_dot_product_attention',
    'aten::matmul',
    'aten::mm',
    'aten::addmm',
    'aten::mul',
    'aten::add',
    'aten::item',
    'cudaStreamSynchronize',
    'cudaLaunchKernel',
]

rows = []
for op in KEY_OPS:
    row = {'Operator': op}
    for label in MODELS:
        if label not in REPORT:
            row[f'{label} ms'] = None
            row[f'{label} %'] = None
            continue
        ms = REPORT[label]['op_time'].get(op, 0.0)
        pct = ms / REPORT[label]['total_ms'] * 100
        row[f'{label} ms'] = round(ms, 2)
        row[f'{label} %'] = round(pct, 2)
    rows.append(row)

df = pd.DataFrame(rows).set_index('Operator')
print('Key Operator Comparison (ms and % of total traced time)')
print('=' * 80)
df

Key Operator Comparison (ms and % of total traced time)


,base ms,base %,control ms,control %,runtime_aware ms,runtime_aware %
Operator,,,,,,
aten::linear,1799.06,4.58,1814.95,4.58,1778.16,4.61
aten::scaled_dot_product_attention,665.38,1.70,692.64,1.75,665.29,1.73
aten::matmul,966.54,2.46,966.19,2.44,958.43,2.49
aten::mm,737.12,1.88,729.04,1.84,726.67,1.89
aten::addmm,596.57,1.52,605.27,1.53,584.28,1.52
aten::mul,530.15,1.35,528.60,1.33,509.40,1.32
aten::add,357.66,0.91,353.32,0.89,342.47,0.89
aten::item,27.95,0.07,27.44,0.07,30.00,0.08
cudaStreamSynchronize,19.47,0.05,18.80,0.05,22.20,0.06


## Step 5: TensorBoard

View traces for all three models side by side in TensorBoard.
Each model's traces are in a separate subdirectory, which TensorBoard uses as the run label.

In [14]:
%reload_ext tensorboard
%tensorboard --logdir outputs/gpu_profiling

## Step 6: Operator-Level Profiling

Runs `profile_operators.py` for each model, producing a bottleneck report and CSV of operator times.
Output is saved to `outputs/operator_profiling/<model>/`.

In [15]:
OP_PROFILE_BASE = PROJECT_ROOT / 'outputs/operator_profiling'

for label, model_path in MODELS.items():
    output_dir = str(OP_PROFILE_BASE / label)
    print(f'\n{"="*60}')
    print(f'Operator profiling: {label}')
    print(f'{"="*60}')
    subprocess.run([
        sys.executable, 'profiling/profile_operators.py',
        '--input_path',     PROFILE_INPUT,
        '--model_name',     model_path,
        '--limit',          str(LIMIT),
        '--batch_size',     str(BATCH_SIZE),
        '--max_new_tokens', str(MAX_NEW_TOKENS),
        '--output_dir',     output_dir,
    ], check=True)



Operator profiling: base
Device : cuda  |  dtype : torch.bfloat16
Output : /workspace/efficient-codegen/outputs/operator_profiling/base


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 392.56it/s]



Warming up (1 example, not profiled)...
Warm-up done.

Profiling 20 example(s) in 1 batch(es) of up to 20...



/venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


[batch 1/1]  batch_size=20  input_len=738  output_len=128  latency=7.390s  tok/s=346.4

Chrome trace saved → /workspace/efficient-codegen/outputs/operator_profiling/base/chrome_trace.json
  View at: ui.perfetto.dev  or  chrome://tracing
TensorBoard trace  → /workspace/efficient-codegen/outputs/operator_profiling/base/tb_trace/worker0.1777155917742.pt.trace.json

Note: CUDA times are zero — reporting CPU times instead. This is normal when CUDA kernels run asynchronously and attribution is unavailable in this PyTorch build.
Operator CSV saved  → /workspace/efficient-codegen/outputs/operator_profiling/base/operators.csv

OPERATOR-LEVEL BOTTLENECK REPORT

Top 30 operators by CPU self-time:

Operator                                                   CUDA ms     CPU ms    Calls  % total
-----------------------------------------------------------------------------------------------
full_generate                                                0.000   7389.694        1    27.2%
decode_remaining

...

[batch 1/1]  batch_size=20  input_len=738  output_len=128  latency=7.669s  tok/s=333.8

Chrome trace saved → /workspace/efficient-codegen/outputs/operator_profiling/control/chrome_trace.json
  View at: ui.perfetto.dev  or  chrome://tracing
TensorBoard trace  → /workspace/efficient-codegen/outputs/operator_profiling/control/tb_trace/worker0.1777156066022.pt.trace.json

Note: CUDA times are zero — reporting CPU times instead. This is normal when CUDA kernels run asynchronously and attribution is unavailable in this PyTorch build.
Operator CSV saved  → /workspace/efficient-codegen/outputs/operator_profiling/control/operators.csv

OPERATOR-LEVEL BOTTLENECK REPORT

Top 30 operators by CPU self-time:

Operator                                                   CUDA ms     CPU ms    Calls  % total
-----------------------------------------------------------------------------------------------
full_generate                                                0.000   7668.361        1    27.2%
decode_

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 560.73it/s]



Warming up (1 example, not profiled)...
Warm-up done.

Profiling 20 example(s) in 1 batch(es) of up to 20...



/venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


[batch 1/1]  batch_size=20  input_len=738  output_len=128  latency=7.808s  tok/s=327.9

Chrome trace saved → /workspace/efficient-codegen/outputs/operator_profiling/runtime_aware/chrome_trace.json
  View at: ui.perfetto.dev  or  chrome://tracing
TensorBoard trace  → /workspace/efficient-codegen/outputs/operator_profiling/runtime_aware/tb_trace/worker0.1777156215064.pt.trace.json

Note: CUDA times are zero — reporting CPU times instead. This is normal when CUDA kernels run asynchronously and attribution is unavailable in this PyTorch build.
Operator CSV saved  → /workspace/efficient-codegen/outputs/operator_profiling/runtime_aware/operators.csv

OPERATOR-LEVEL BOTTLENECK REPORT

Top 30 operators by CPU self-time:

Operator                                                   CUDA ms     CPU ms    Calls  % total
-----------------------------------------------------------------------------------------------
full_generate                                                0.000   7807.671        

## Step 7: Operator Bottleneck Reports

In [16]:
for label in MODELS:
    report = OP_PROFILE_BASE / label / 'bottleneck_report.txt'
    if report.exists():
        print(f'\n{"="*60}')
        print(f'  {label}')
        print(f'{"="*60}')
        print(report.read_text(encoding='utf-8')[:3000])
    else:
        print(f'No report found for {label}')



  base
OPERATOR-LEVEL BOTTLENECK REPORT

Top 30 operators by CPU self-time:

Operator                                                   CUDA ms     CPU ms    Calls  % total
-----------------------------------------------------------------------------------------------
full_generate                                                0.000   7389.694        1    27.2%
decode_remaining                                             0.000   6367.912      122    23.4%
cudaLaunchKernel                                             0.000   1577.710   177664     5.8%
aten::linear                                                 0.000   1524.913    25216     5.6%
aten::matmul                                                 0.000    710.307    14592     2.6%
aten::scaled_dot_product_attention                           0.000    682.632     3584     2.5%
aten::mul                                                    0.000    528.391    32768     1.9%
aten::mm                                                  

## Step 8: Operator CSV Comparison

Loads `operators.csv` from each model run and shows a side-by-side comparison of top operators by CPU time.

In [17]:
import pandas as pd

op_dfs = {}
for label in MODELS:
    csv_path = OP_PROFILE_BASE / label / 'operators.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        op_dfs[label] = df.set_index('operator') if 'operator' in df.columns else df
    else:
        print(f'No operators.csv for {label}')

if op_dfs:
    col = 'cpu_time_ms'
    top_ops = (
        pd.concat({k: v[col] for k, v in op_dfs.items() if col in v.columns}, axis=1)
        .fillna(0)
        .assign(total=lambda d: d.sum(axis=1))
        .sort_values('total', ascending=False)
        .drop(columns='total')
        .head(20)
    )
    print('Top 20 operators by CPU time (ms)')
    print('=' * 60)
    display(top_ops)


ValueError: No objects to concatenate